# Final Comparison and report
Pull all experiment reports, compare models, perturbations, UQ methods, and XAI impact.

In [1]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.visualization import  save_figure

config = load_config()
reports_path = config['paths']['results_reports']

# Load all reports
def load_report(filename):
    path = os.path.join(reports_path, filename)
    with open(path, 'r') as f:
        return json.load(f)

def get_result(report):
    if 'stage1' in report:
        return report['stage1']['result']
    elif 'stage2' in report:
        return report['stage2']['result']
    else:
        return None

# List all report files
report_files = [f for f in os.listdir(reports_path) if f.endswith('.json')]
for f in sorted(report_files):
    print(f)

TC_01_Random_Forest_Clean_Baseline.json
TC_02_XGBoost_Clean_Baseline.json
TC_03_FCNN_Golden_Baseline.json
TC_04_FCNN_Fault_Values_10_Fraction.json
TC_04_FCNN_Fault_Values_20_Fraction.json
TC_04_FCNN_Fault_Values_5_Fraction.json
TC_05_FCNN_Class_Imbalance_10_Fraction.json
TC_05_FCNN_Class_Imbalance_30_Fraction.json
TC_05_FCNN_Class_Imbalance_50_Fraction.json
TC_06_FCNN_noise_Inject_10_Fraction.json
TC_06_FCNN_noise_Inject_20_Fraction.json
TC_06_FCNN_noise_Inject_50_Fraction.json
TC_07_FCNN_Data_Size_25_Fraction.json
TC_07_FCNN_Data_Size_50_Fraction.json
TC_07_FCNN_Data_Size_75_Fraction.json
TC_08_FCNN_Training_With_Hyperparameter_LR_0.0001_BS_128_Variations.json
TC_08_FCNN_Training_With_Hyperparameter_LR_0.0001_BS_32_Variations.json
TC_08_FCNN_Training_With_Hyperparameter_LR_0.01_BS_128_Variations.json
TC_08_FCNN_Training_With_Hyperparameter_LR_0.01_BS_32_Variations.json
TC_09_FCNN_Seed_Variation_0.json
TC_09_FCNN_Seed_Variation_1.json
TC_09_FCNN_Seed_Variation_2.json
TC_09_FCNN_Seed_Va

### Comparison 1: Model Architectures on Clean Data

In [2]:
tc01 = load_report('TC_01_Random_Forest_Clean_Baseline.json')
tc02 = load_report('TC_02_XGBoost_Clean_Baseline.json')
tc03 = load_report('TC_03_FCNN_Golden_Baseline.json')
tc10 = load_report('TC_10_Simple_CNN_Golden_Baseline.json')
tc11 = load_report('TC_11_ResNet_Golden_Baseline.json')

rf = tc01['stage1']['result']
xgb = tc02['stage1']['result']
fcnn = tc03['stage1']['result']
cnn = tc10['stage2']['result']
resnet = tc11['stage2']['result']

# Bar Chart
models = ['RF', 'XGBoost', 'FCNN']
accuracies = [rf['accuracy'], xgb['accuracy'], fcnn['accuracy']]
consistencies = [rf['consistency']['mean'], xgb['consistency']['mean'], fcnn['consistency']['mean']]
de_uncertainty = [rf['de_uncertainty']['mean'], xgb['de_uncertainty']['mean'], fcnn['de_uncertainty']['mean']]

mc_uncertainty = [0, 0, fcnn['mc_uncertainty']['mean']]
precision_values = [
    rf['classification_report']['macro avg']['precision'],
    xgb['classification_report']['macro avg']['precision'],
    fcnn['classification_report']['macro avg']['precision']
]
recall_values = [
    rf['classification_report']['macro avg']['recall'],
    xgb['classification_report']['macro avg']['recall'],
    fcnn['classification_report']['macro avg']['recall']
]
f1_values = [
    rf['classification_report']['macro avg']['f1-score'],
    xgb['classification_report']['macro avg']['f1-score'],
    fcnn['classification_report']['macro avg']['f1-score']
]

metrics = {
'accuracies':accuracies,
'consistencies':consistencies,
'precision':precision_values,
'recall':recall_values,
'f1':f1_values,
'de_uncertainty':de_uncertainty,
'mc_uncertainty':mc_uncertainty
}

x = np.arange(len(models))
width = 0.15
multiplier = 0
color_set = sns.color_palette("pastel", len(metrics))

fig, ax = plt.subplots(figsize = (12, 6))

for (metrics_name, values), color in zip(metrics.items(), color_set):
    offset = width * multiplier
    ax.bar(x + offset, values, width, color = color, label = metrics_name, linewidth= 0.5)
    multiplier += 1

ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison - Baseline', fontweight = 'bold', pad = 20)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.1)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.yaxis.grid(True, linestyle='--', alpha = 0.6)
ax.set_axisbelow(True)

ax.legend(loc= 'upper center', bbox_to_anchor = (0.5, -0.12), ncol = 2, frameon = False)

plt.tight_layout()
save_figure(fig, 'comparison_model_baseline.png', config)





Saved: results/figures\comparison_model_baseline.png


### Comparison 2: Perturbation Impact on FCNN

In [3]:
mv_files = sorted([f for f in report_files if 'Fault_Values' in f])
ci_files = sorted([f for f in report_files if 'Class_Imbalance' in f])
ni_files = sorted([f for f in report_files if 'noise_Inject' in f])

# ALl perturbation reports
all_perturbation_files = mv_files + ci_files + ni_files

perturbation_name = []
perturbation_accuracies = []
perturbation_precision = []
perturbation_recall = []
perturbation_f1 = []

for f in all_perturbation_files:
    r = get_result(load_report(f))
    name = f.replace('.json', '').replace('TC_0', '').replace('_Fraction', '').replace('FCNN_', '')[:25]
    cr = r['classification_report']
    perturbation_name.append(name)
    perturbation_accuracies.append(r['accuracy'])
    perturbation_precision.append(round(cr['macro avg']['precision'], 4))
    perturbation_recall.append(round(cr['macro avg']['recall'], 4))
    perturbation_f1.append(round(cr['macro avg']['f1-score'], 4))

baseline_acc = fcnn['accuracy']

fig, ax = plt.subplots(figsize = (14, 6))
x = np.arange(len(perturbation_name))
width = 0.2

colors = sns.color_palette("pastel", 4)
ax.bar(x - width * 1.5, perturbation_accuracies, width, label= 'Accuracy', color= colors[0])
ax.bar(x - width * 0.5, perturbation_precision, width, label= 'Precision', color= colors[1])
ax.bar(x + width * 0.5, perturbation_recall, width, label= 'Recall', color= colors[2])
ax.bar(x + width * 1.5, perturbation_f1, width, label= 'F1', color= colors[3])
ax.axhline(y = baseline_acc, color = 'red', linestyle = '--', label= 'Baseline accuracy')
ax.axhline(y = 0, color = 'red', linestyle = '--', label= 'Baseline')

ax.set_xticks(x)
ax.set_xticklabels(perturbation_name, rotation = 20, ha = 'right')
ax.set_ylabel('Score')
ax.set_title('Perturbation Impact on Model Performance', fontweight = 'bold', pad = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_axisbelow(True)
ax.legend(loc = 'upper center', bbox_to_anchor = (0.5, -0.25), ncol = 3, frameon = False)
ax.yaxis.grid(True, linestyle='--', alpha = 0.6)
ax.set_axisbelow(True)
plt.tight_layout()
save_figure(fig, 'comparison_perturbation_accuracy.png', config)


Saved: results/figures\comparison_perturbation_accuracy.png


### Comparison 3: UQ Methods Response to Perturbation - MC Dropout vs ENSEMBLES

In [4]:
mc_uncertainties = []
de_uncertainties = []

for f in all_perturbation_files:
    r = get_result(load_report(f))
    mc_uncertainties.append(r['mc_uncertainty']['mean'])
    de_uncertainties.append(r['de_uncertainty']['mean'])

fig, ax = plt.subplots(figsize = (14, 6))
x = np.arange(len(perturbation_name))
width = 0.35

ax.bar(x - width / 2, mc_uncertainties, width, label= 'MC Dropout', color= 'steelblue')
ax.bar(x + width / 2, de_uncertainties, width, label= 'Deep Ensembles', color= 'coral')

ax.axhline(y = fcnn['mc_uncertainty']['mean'], color = 'steelblue', linestyle = '--', label= 'MC baseline', alpha = 0.5)
ax.axhline(y = fcnn['de_uncertainty']['mean'], color = 'coral', linestyle = '--', label= 'DE baseline', alpha = 0.5)

ax.set_xticks(x)
ax.set_xticklabels(perturbation_name, rotation = 20, ha = 'right')
ax.set_ylabel('Mean Uncertainty')
ax.set_title('Uncertainty Response to Perturbation', fontweight = 'bold', pad = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_axisbelow(True)
ax.legend(loc = 'upper center', bbox_to_anchor = (0.5, -0.25), ncol = 3, frameon = False)
ax.yaxis.grid(True, linestyle='--', alpha = 0.6)
ax.set_axisbelow(True)
plt.tight_layout()
save_figure(fig, 'comparison_uncertainty_accuracy.png', config)


Saved: results/figures\comparison_uncertainty_accuracy.png


### Comparison 4: Core Finding - UQ Only vs UQ + XAI

In [5]:
all_experiment_files = sorted([f for f in report_files if any(tc in f for tc in ['TC_04', 'TC_05', 'TC_06', 'TC_07', 'TC_08', 'TC_09'])])

mc_improvements = []
de_improvements = []
experiment_names = []

for f in all_experiment_files:
    r = get_result(load_report(f))
    name = f.replace('.json', '').replace('TC_0', '').replace('_Fraction', '').replace('_Variations', '')[:25]
    mc = r.get('mc_vs_mc_plus_xai', 0)
    de = r.get('de_vs_de_plus_xai', 0)
    experiment_names.append(name)
    mc_improvements.append(mc)
    de_improvements.append(de)

fig, ax = plt.subplots(figsize = (16, 6))
x = np.arange(len(experiment_names))
width = 0.35

ax.bar(x - width / 2, mc_improvements, width, label= 'MC + XAI Improvement', color= 'steelblue')
ax.bar(x + width / 2, de_improvements, width, label= 'DE + XAI Improvement', color= 'coral')

ax.axhline(y= 0, color = 'black', linestyle = '--', linewidth = 0.5)


ax.set_xticks(x)
ax.set_xticklabels(experiment_names, rotation = 20, ha = 'right')
ax.set_ylabel('GREEN Accuracy Improvement')
ax.set_title('Does Adding XAI Improve Flagging?', fontweight = 'bold', pad = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_axisbelow(True)
ax.legend(loc = 'upper center', bbox_to_anchor = (0.5, -0.25), ncol = 3, frameon = False)
ax.yaxis.grid(True, linestyle='--', alpha = 0.6)
ax.set_axisbelow(True)
plt.tight_layout()
save_figure(fig, 'comparison_uq_vs_uq_xai_accuracy.png', config)

Saved: results/figures\comparison_uq_vs_uq_xai_accuracy.png


### Comparison 5: Tabular vs Image Comparison

In [6]:
all_models = ['RF', 'XGBoost', 'FCNN', 'CNN', 'ResNet']
all_accuracies = [rf['accuracy'], xgb['accuracy'], fcnn['accuracy'], cnn['accuracy'],resnet['accuracy']]

fig, ax = plt.subplots(figsize = (10, 6))
colors = ['steelblue', 'steelblue', 'steelblue', 'coral', 'coral']
bars = ax.bar(all_models, all_accuracies, color= colors)

ax.set_ylabel('Accuracy')
ax.set_title('Tabular vs Image Model Accuracy', fontweight = 'bold', pad = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_axisbelow(True)
ax.set_axisbelow(True)

legend_elements = [Patch(facecolor= 'steelblue', label = 'Tabular'), Patch(facecolor='coral', label = 'Image')]
ax.legend(handles = legend_elements, frameon = False)
plt.tight_layout()
save_figure(fig, 'comparison_tabular_vs_image.png', config)



Saved: results/figures\comparison_tabular_vs_image.png
